In [12]:
import random
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
import numpy as np
import torch.utils.data
import cv2
import torchvision.models.segmentation
import torch
import os

In [13]:
batchSize=2
imageSize=[600,600]
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')   # train on the GPU or on the CPU, if a GPU is not available
trainDir="./Datasets/LabPics Chemistry/Train"

imgs=[]
for pth in os.listdir(trainDir):
    imgs .append(trainDir+"/"+pth +"//")
device

device(type='cpu')

In [14]:
imgs[:10]

['./datasets/LabPics Chemistry/Train/959Train//',
 './datasets/LabPics Chemistry/Train/3951Train//',
 './datasets/LabPics Chemistry/Train/2643Train//',
 './datasets/LabPics Chemistry/Train/2213Train//',
 './datasets/LabPics Chemistry/Train/1595Train//',
 './datasets/LabPics Chemistry/Train/4797Train//',
 './datasets/LabPics Chemistry/Train/3052Train//',
 './datasets/LabPics Chemistry/Train/3402Train//',
 './datasets/LabPics Chemistry/Train/3117Train//',
 './datasets/LabPics Chemistry/Train/4328Train//']

In [15]:
def loadData():
    batch_Imgs=[]
    batch_Data=[]# load images and masks
    for i in range(batchSize):
        idx=random.randint(0,len(imgs)-1)
        img = cv2.imread(os.path.join(imgs[idx], "Image.jpg"))
        img = cv2.resize(img, imageSize, cv2.INTER_LINEAR)
        maskDir=os.path.join(imgs[idx], "Vessels")
        masks=[]
        for mskName in os.listdir(maskDir):
            vesMask = (cv2.imread(maskDir+'/'+mskName, 0) > 0).astype(np.uint8)  # Read vesse instance mask
            vesMask=cv2.resize(vesMask,imageSize,cv2.INTER_NEAREST)
            masks.append(vesMask)# get bounding box coordinates for each mask
        num_objs = len(masks)
        if num_objs==0: return loadData() # if image have no objects just load another image
        boxes = torch.zeros([num_objs,4], dtype=torch.float32)
        for i in range(num_objs):
            x,y,w,h = cv2.boundingRect(masks[i])
            boxes[i] = torch.tensor([x, y, x+w, y+h])
        masks = torch.as_tensor(masks, dtype=torch.uint8)
        img = torch.as_tensor(img, dtype=torch.float32)
        data = {}
        data["boxes"] =  boxes
        data["labels"] =  torch.ones((num_objs,), dtype=torch.int64)   # there is only one class
        data["masks"] = masks
        batch_Imgs.append(img)
        batch_Data.append(data)  # load images and masks
    batch_Imgs = torch.stack([torch.as_tensor(d) for d in batch_Imgs], 0)
    batch_Imgs = batch_Imgs.swapaxes(1, 3).swapaxes(2, 3)
    return batch_Imgs, batch_Data

In [16]:
model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)  # load an instance segmentation model pre-trained pre-trained on COCO
in_features = model.roi_heads.box_predictor.cls_score.in_features  # get number of input features for the classifier
model.roi_heads.box_predictor = FastRCNNPredictor(in_features,num_classes=2)  # replace the pre-trained head with a new one
model.to(device)# move model to the right devic

optimizer = torch.optim.AdamW(params=model.parameters(), lr=1e-5)
model.train()

MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [17]:
for i in range(10001):
            images, targets = loadData()
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            optimizer.zero_grad()
            loss_dict = model(images, targets)

            losses = sum(loss for loss in loss_dict.values())
            losses.backward()
            optimizer.step()
            print(i,'loss:', losses.item())
            if i%500==0:
                torch.save(model.state_dict(), str(i)+".torch")

0 loss: 256.28033447265625
1 loss: 209.8935089111328
2 loss: 61.34748458862305
3 loss: 20.72884178161621
4 loss: 25.351076126098633
5 loss: 20.705337524414062
6 loss: 25.459747314453125
7 loss: 21.245769500732422
8 loss: 32.92937088012695
9 loss: 23.27105140686035
10 loss: 35.375186920166016
11 loss: 29.73285484313965
12 loss: 28.111669540405273
13 loss: 12.96098804473877
14 loss: 34.848506927490234
15 loss: 11.692169189453125
16 loss: 16.814912796020508
17 loss: 16.308961868286133
18 loss: 18.962345123291016
19 loss: 6.83730411529541
20 loss: 11.59646224975586
21 loss: 9.68680191040039
22 loss: 79.28791809082031
23 loss: 9.195853233337402
24 loss: 32.07895278930664
25 loss: 8.642166137695312
26 loss: 6.178889274597168
27 loss: 5.8801774978637695
28 loss: 8.599544525146484
29 loss: 8.078392028808594
30 loss: 6.580582141876221
31 loss: 9.400142669677734
32 loss: 8.654580116271973


KeyboardInterrupt: 